### Denoising figures

#### Figure 1

In [ ]:
import os
from pathlib import Path
DATA_ROOT = Path(os.environ.get("VF_DATA_ROOT", "/path/to/vf_oct_pairs"))   # see config/env.example.sh

# ==========================================
# Figure: single patient example
# Saves 4 separate figures:
#   1. Raw VF
#   2. Denoised VF (NAFNet)
#   3. Predicted RNFL (denoised VF → vf_to_rnfl_skip model)
#   4. Ground Truth RNFL
# Direction-oriented: left-eye RNFL flipped LR (matching training convention)
# ==========================================

import importlib.util
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1 import make_axes_locatable
from pathlib import Path

# ---- Paths ----
RAW_PATH  = DATA_ROOT / "original_below350"
NAF_PATH  = DATA_ROOT / "denoised_NAFNet_models/denoised_NAFNet_Advanced"
CKPT_PATH = Path("checkpoints/Skip16_NAFNet_Tail_best.pth")
OUT_DIR   = Path("figures_out")
OUT_DIR.mkdir(exist_ok=True)

Z_DIM         = 32
SKIP_CHANNELS = 16
IMG_SIZE      = 200
VF_VMIN, VF_VMAX = -38.0, 26.0
RNFL_VMAX = 350.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Load vf_to_rnfl_skip module (model classes + normalization helpers) ----
spec   = importlib.util.spec_from_file_location("vf_to_rnfl_skip", "vf_to_rnfl_skip.py")
vfskip = importlib.util.module_from_spec(spec)
spec.loader.exec_module(vfskip)

# ---- Build VF2RNFL model (skip architecture) & load checkpoint ----
decoder = vfskip.RNFL_Decoder(z_dim=Z_DIM, img_size=IMG_SIZE).to(device)
model   = vfskip.VF2RNFL_Model(
    decoder,
    z_dim=Z_DIM,
    dropout=0.1,
    img_size=IMG_SIZE,
    skip_channels=SKIP_CHANNELS
).to(device)

ckpt  = torch.load(CKPT_PATH, map_location=device, weights_only=False)
state = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
model.load_state_dict(state)
model.eval()
print("VF2RNFL (skip) model loaded:", CKPT_PATH.name)

# ---- Pick one patient common to both raw and NAFNet ----
raw_names = set(f.name for f in RAW_PATH.glob("*.npz"))
naf_names = set(f.name for f in NAF_PATH.glob("*.npz"))
common    = sorted(raw_names & naf_names)
print(f"Found {len(common)} common files")

# Change PATIENT_IDX to select a different patient
PATIENT_IDX = 42
fname   = common[PATIENT_IDX]
is_left = "_0_" in fname
print(f"Selected: {fname}  ({'Left' if is_left else 'Right'} eye)")

# ---- Load data ----
raw_data = np.load(RAW_PATH / fname, allow_pickle=True)
naf_data = np.load(NAF_PATH / fname, allow_pickle=True)

vf_raw  = raw_data["td"].astype(np.float32)     # (52,) dB
vf_den  = naf_data["td"].astype(np.float32)     # (52,) dB  NAFNet-denoised
rnfl_gt = raw_data["rnflt"].astype(np.float32)  # (200,200) µm

# Apply left-eye flip to RNFL (matches training convention)
if is_left:
    rnfl_gt = vfskip.flip_rnfl_image(rnfl_gt)

# ---- Predict RNFL from denoised VF ----
# Pipeline: vf_den → normalize [-1,1] → 8×9 grid → (1,1,8,9) → model → ×350 µm
vf_den_norm = vfskip.normalize_vf(vf_den)               # (52,) in [-1, 1]
vf_den_grid = vfskip.map_vf_to_8x9(vf_den_norm)         # (8, 9)
vf_tensor   = torch.tensor(vf_den_grid).unsqueeze(0).unsqueeze(0).to(device)  # (1,1,8,9)

with torch.no_grad():
    pred_norm, _ = model(vf_tensor)
    rnfl_pred = pred_norm.cpu().squeeze().numpy() * RNFL_VMAX  # (200,200) µm

print(f"GT RNFL    : {rnfl_gt.min():.1f}–{rnfl_gt.max():.1f} µm")
print(f"Pred RNFL  : {rnfl_pred.min():.1f}–{rnfl_pred.max():.1f} µm")
print(f"MAE        : {np.mean(np.abs(rnfl_pred - rnfl_gt)):.1f} µm")


# ============================================================
# Helpers
# ============================================================
def _gen_vfmat(tds):
    """Map 52 TD values into an 8×9 grid (24-2 pattern, blind spot as NaN)."""
    mat   = np.zeros([8, 9])
    nulls = {0,1,2,7,8,9,10,17,18,34,43,45,54,55,62,63,64,65,70,71}
    k = 0
    for i in range(8):
        for j in range(9):
            if i * 9 + j not in nulls:
                mat[i][j] = tds[k]
                k += 1
    mat[3][7] = np.nan   # blind spot (superior)
    mat[4][7] = np.nan   # blind spot (inferior)
    return mat


def save_vf_fig(vf_vec, filepath, title=None):
    """Save a standalone VF heatmap (8×9, bwr_r, fixed −38…26 dB scale)."""
    mat  = _gen_vfmat(vf_vec)
    cmap = plt.get_cmap("bwr_r").copy()
    cmap.set_bad("lightgray")
    divnorm = colors.TwoSlopeNorm(vmin=VF_VMIN, vcenter=0, vmax=VF_VMAX)

    fig, ax = plt.subplots(figsize=(4, 3.5))
    im = ax.imshow(mat, cmap=cmap, norm=divnorm, aspect="auto")
    ax.axis("off")
    ax.add_patch(Rectangle((6.5, 2.5), 1, 2,
                            fill=False, edgecolor="black", lw=0.8))

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("bottom", size="5%", pad=0.1)
    cbar = plt.colorbar(im, orientation="horizontal", cax=cax,
                        ticks=[VF_VMIN, 0, VF_VMAX])
    cbar.ax.tick_params(labelsize=9)
    cbar.set_label("TD (dB)", fontsize=9)

    if title:
        ax.set_title(title, fontsize=12, fontweight="bold")

    plt.tight_layout()
    plt.savefig(filepath, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {filepath}")


def save_rnfl_fig(rnfl_map, filepath, title=None):
    """Save a standalone RNFL map (jet, 0–350 µm, origin=lower)."""
    fig, ax = plt.subplots(figsize=(4, 3.5))
    im = ax.imshow(rnfl_map, cmap="jet", vmin=0, vmax=RNFL_VMAX, origin="lower")
    ax.axis("off")
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=9)
    cbar.set_label("RNFL (µm)", fontsize=9)

    if title:
        ax.set_title(title, fontsize=12, fontweight="bold")

    plt.tight_layout()
    plt.savefig(filepath, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {filepath}")


# ============================================================
# Generate & save the 4 figures
# ============================================================
pat_id = fname.replace(".npz", "")
print(f"\nGenerating 4 figures for: {pat_id}\n")

save_vf_fig  (vf_raw,    OUT_DIR / f"{pat_id}_01_raw_vf.png",         title="Raw VF")
save_vf_fig  (vf_den,    OUT_DIR / f"{pat_id}_02_denoised_vf.png",    title="Denoised VF (NAFNet)")
save_rnfl_fig(rnfl_pred, OUT_DIR / f"{pat_id}_03_predicted_rnfl.png", title="Predicted RNFL")
save_rnfl_fig(rnfl_gt,   OUT_DIR / f"{pat_id}_04_gt_rnfl.png",        title="Ground Truth RNFL")

print(f"\nDone. Figures in: {OUT_DIR.resolve()}")


#### Figure 2

In [ ]:
import os
from pathlib import Path
DATA_ROOT = Path(os.environ.get("VF_DATA_ROOT", "/path/to/vf_oct_pairs"))   # see config/env.example.sh


# ==========================================
# Grid Figure: VF Denoising Comparison
#
# Row 0  -- VF map:  Raw | N2N | NAFNet | CNN-AE | CNN-VAE
# Row 1  -- |Diff|:  (blank) | |N2N-Raw| | |NAFNet-Raw| | |CNN-AE-Raw| | |CNN-VAE-Raw|
# ==========================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import colors
from matplotlib.patches import Rectangle
from pathlib import Path

# ---- Paths ----
RAW_PATH = DATA_ROOT / "original_below350"

DENOISED_PATHS = {
    "NAFNet":    DATA_ROOT / "denoised_NAFNet_models/denoised_NAFNet_Advanced",
    "N2N":                        DATA_ROOT / "denoised_n2n_FINAL",
    "CNN-AE":                     DATA_ROOT / "denoised_TwoStage_Final/denoised_cnn_ae",
    "CNN-VAE":                    DATA_ROOT / "denoised_TwoStage_Final/denoised_cnn_vae",
}

OUT_DIR = Path("figures_out")
OUT_DIR.mkdir(exist_ok=True)

VF_VMIN, VF_VMAX = -38.0, 26.0
DIFF_VMAX = 10.0

# ---- Common files ----
common = set(f.name for f in RAW_PATH.glob("*.npz"))
for p in DENOISED_PATHS.values():
    common &= set(f.name for f in p.glob("*.npz"))
common = sorted(common)
print(f"Common files: {len(common)}")

PATIENT_IDX = 42
fname = common[PATIENT_IDX]
print(f"Patient: {fname}")

# ---- Load VF data ----
vf_raw  = np.load(RAW_PATH / fname, allow_pickle=True)["td"].astype(np.float32)
vf_dict = {}
for name, p in DENOISED_PATHS.items():
    raw = np.load(p / fname, allow_pickle=True)["td"].astype(np.float32)
    vf_dict[name] = raw.squeeze() if raw.ndim == 2 else raw

# ---- 52-point -> 8x9 grid ----
def _gen_vfmat(tds):
    mat   = np.zeros([8, 9])
    nulls = {0,1,2,7,8,9,10,17,18,34,43,45,54,55,62,63,64,65,70,71}
    k = 0
    for i in range(8):
        for j in range(9):
            if i * 9 + j not in nulls:
                mat[i][j] = tds[k]
                k += 1
    mat[3][7] = np.nan
    mat[4][7] = np.nan
    return mat

# ---- Colormaps ----
vf_cmap = plt.get_cmap("bwr_r").copy()
vf_cmap.set_bad("lightgray")
vf_norm = colors.TwoSlopeNorm(vmin=VF_VMIN, vcenter=0, vmax=VF_VMAX)

import numpy as np
from matplotlib import colors
import matplotlib.pyplot as plt

# --- |Diff| colormap: 0 -> white, then smooth Reds ---
base = plt.get_cmap("Reds", 256)
cols = base(np.linspace(0, 1, 256))
cols[0] = [1, 1, 1, 1]  # force exact white at the bottom
diff_cmap = colors.ListedColormap(cols)
diff_cmap.set_bad("lightgray")

diff_norm = colors.PowerNorm(gamma=0.7, vmin=0, vmax=DIFF_VMAX)
# ---- Layout: 2 rows x 5 cols ----
COL_LABELS = ["Raw"] + list(DENOISED_PATHS.keys())
ROW_LABELS = ["TD (dB)", "|ΔTD| (dB)"]
n_cols, n_rows = len(COL_LABELS), 2

fig = plt.figure(figsize=(2.8 * n_cols, 3.0 * n_rows + 0.5))
gs  = gridspec.GridSpec(
    n_rows, n_cols, figure=fig,
    hspace=0.20, wspace=0.05,
    top=0.91, bottom=0.09,
    left=0.07, right=0.98,
)
axs = [[fig.add_subplot(gs[r, c]) for c in range(n_cols)] for r in range(n_rows)]

# Column titles
for j, label in enumerate(COL_LABELS):
    axs[0][j].set_title(label, fontsize=11, fontweight="bold", pad=5)

# Row labels (left side)
for i, rl in enumerate(ROW_LABELS):
    pos = axs[i][0].get_position()
    fig.text(0.005, (pos.y0 + pos.y1) / 2, rl,
             va="center", ha="left", fontsize=10, fontweight="bold", rotation=90)

def _plot_vf(ax, mat, cmap, norm, blind_spot=True):
    ax.imshow(mat, cmap=cmap, norm=norm, aspect="auto")
    if blind_spot:
        ax.add_patch(Rectangle((6.5, 2.5), 1, 2,
                                fill=False, edgecolor="black", lw=0.7))
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for sp in ax.spines.values():
        sp.set_visible(False)

# Row 0: VF maps
_plot_vf(axs[0][0], _gen_vfmat(vf_raw), vf_cmap, vf_norm)
for j, (name, vf_den) in enumerate(vf_dict.items(), start=1):
    _plot_vf(axs[0][j], _gen_vfmat(vf_den), vf_cmap, vf_norm)

# Row 1: |Diff| maps (Raw col is blank)
axs[1][0].set_visible(False)
for j, (name, vf_den) in enumerate(vf_dict.items(), start=1):
    diff = np.abs(vf_den - vf_raw)
    _plot_vf(axs[1][j], _gen_vfmat(diff), diff_cmap, diff_norm, blind_spot=False)

# ---- Shared colorbars ----
cax_vf = fig.add_axes([0.07, 0.03, 0.55, 0.018])
sm_vf  = plt.cm.ScalarMappable(cmap=vf_cmap, norm=vf_norm);  sm_vf.set_array([])
cb_vf  = fig.colorbar(sm_vf, cax=cax_vf, orientation="horizontal",
                       ticks=[VF_VMIN, 0, VF_VMAX])
cb_vf.ax.tick_params(labelsize=9);  cb_vf.set_label("TD (dB)", fontsize=9)

cax_d  = fig.add_axes([0.66, 0.03, 0.32, 0.018])
sm_d   = plt.cm.ScalarMappable(cmap=diff_cmap, norm=diff_norm);  sm_d.set_array([])
cb_d   = fig.colorbar(sm_d, cax=cax_d, orientation="horizontal",
                       ticks=[0, DIFF_VMAX / 2, DIFF_VMAX])
cb_d.ax.tick_params(labelsize=9);  cb_d.set_label("|ΔTD| (dB)", fontsize=9)

# ---- Save ----
pat_id  = fname.replace(".npz", "")
outpath = OUT_DIR / f"{pat_id}_vf_grid_methods.png"
plt.savefig(outpath, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {outpath.resolve()}")


#### Figure 3

In [ ]:
import os
from pathlib import Path
DATA_ROOT = Path(os.environ.get("VF_DATA_ROOT", "/path/to/vf_oct_pairs"))   # see config/env.example.sh

# ==========================================
# Figure: raw mask with contour boundary
#
# VF null cells = WHITE
# VF not stretched (square cells)
# RNFL masked region = GREYISH (RAW-derived keep mask)
# Disc boundary = SAME in every row (contour from RAW mask at final resolution)
# Full-width GT row
# ==========================================

import importlib.util
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import colors
from pathlib import Path
import cv2

# -----------------------
# PATHS / SETTINGS
# -----------------------
RAW_PATH = DATA_ROOT / "original_below350"
VF_PATHS = {
    "Raw":        RAW_PATH,
    "NAFNet":     DATA_ROOT / "denoised_NAFNet_models/denoised_NAFNet_Advanced",
    "Noise2Void": DATA_ROOT / "denoised_n2v",
    "CNN-VAE":    DATA_ROOT / "denoised_TwoStage_Final/denoised_cnn_vae",
}

CKPTS = {
    "Raw":        Path("checkpoints/Skip24_Raw_All_best.pth"),
    "NAFNet":     Path("checkpoints/Skip24_NAFNet_All_best.pth"),
    "Noise2Void": Path("checkpoints/Skip24_N2V_All_best.pth"),
    "CNN-VAE":    Path("checkpoints/Skip24_CNN_VAE_All_best.pth"),
}

OUT_DIR = Path("figures_out")
OUT_DIR.mkdir(exist_ok=True)

PATIENT_IDX = 102



# Defaults if ckpt lacks args
DEFAULT_Z_DIM = 32
DEFAULT_SKIP_CH = 24
DEFAULT_IMG_SIZE = 200

VF_VMIN, VF_VMAX = -38.0, 26.0
RNFL_VMAX = 350.0

VF_CMAP_NAME = "bwr_r"
RNFL_CMAP_NAME = "turbo"

# Colors
RNFL_MASK_GREY = (0.80, 0.80, 0.80, 1.0)   # masked fill
VF_NULL_WHITE  = (1.00, 1.00, 1.00, 1.0)   # VF null cells
BOUNDARY_COLOR = (0.08, 0.08, 0.08)        # dark grey/near-black

BOUNDARY_LW = 0.85  # consistent thin boundary

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -----------------------
# Load vf_to_rnfl_skip.py as module
# -----------------------
spec = importlib.util.spec_from_file_location("vf_to_rnfl_skip", "vf_to_rnfl_skip.py")
vfskip = importlib.util.module_from_spec(spec)
spec.loader.exec_module(vfskip)

# -----------------------
# Helpers
# -----------------------
def _no_ticks(ax):
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for sp in ax.spines.values():
        sp.set_visible(False)

def get_common_filenames(vf_paths_dict):
    common = set(f.name for f in list(vf_paths_dict.values())[0].glob("*.npz"))
    for p in vf_paths_dict.values():
        common &= set(f.name for f in p.glob("*.npz"))
    return sorted(common)

def vf52_to_8x9(tds_52):
    mat = np.full((8, 9), np.nan, dtype=np.float32)  # NaN -> WHITE
    nulls = {0,1,2,7,8,9,10,17,18,34,43,45,54,55,62,63,64,65,70,71}
    k = 0
    for i in range(8):
        for j in range(9):
            if i * 9 + j not in nulls:
                mat[i, j] = tds_52[k]
                k += 1
    mat[3, 7] = np.nan
    mat[4, 7] = np.nan
    return mat

def load_model(ckpt_path: Path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    ckpt_args = None
    if isinstance(ckpt, dict) and "args" in ckpt and isinstance(ckpt["args"], dict):
        ckpt_args = ckpt["args"]

    z_dim = int(ckpt_args.get("z_dim", DEFAULT_Z_DIM)) if ckpt_args else DEFAULT_Z_DIM
    skip_ch = int(ckpt_args.get("skip_channels", DEFAULT_SKIP_CH)) if ckpt_args else DEFAULT_SKIP_CH
    img_size = int(ckpt_args.get("img_size", DEFAULT_IMG_SIZE)) if ckpt_args else DEFAULT_IMG_SIZE
    dropout = float(ckpt_args.get("dropout", 0.1)) if ckpt_args else 0.1

    decoder = vfskip.RNFL_Decoder(z_dim=z_dim, img_size=img_size).to(device)
    model = vfskip.VF2RNFL_Model(
        decoder,
        z_dim=z_dim,
        dropout=dropout,
        img_size=img_size,
        skip_channels=skip_ch
    ).to(device)

    state = ckpt["model_state_dict"] if (isinstance(ckpt, dict) and "model_state_dict" in ckpt) else ckpt
    model.load_state_dict(state, strict=True)
    model.eval()

    return model, dict(z_dim=z_dim, skip_ch=skip_ch, img_size=img_size)

def predict_rnfl_um(model, vf_vec_52):
    vf_norm = vfskip.normalize_vf(vf_vec_52)
    vf_grid = vfskip.map_vf_to_8x9(vf_norm)
    vf_t = torch.tensor(vf_grid, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        pred, _ = model(vf_t)
    return (pred.cpu().squeeze().numpy() * RNFL_VMAX).astype(np.float32)

def get_keep_mask_from_raw(raw_rnflt_um_2d):
    """
    Keep mask from RAW RNFLT.
    Masks ONLY regions where raw==0 (disc hole / scan boundary).
    Returns keep_mask_256: True=KEEP, False=MASKED
    """
    img_size = 256
    kernel = np.ones((4, 4), np.uint8)
    img_rs = cv2.resize(raw_rnflt_um_2d.astype(np.float32), (img_size, img_size))
    img_rs = cv2.morphologyEx(img_rs, cv2.MORPH_CLOSE, kernel)
    keep_mask_256 = img_rs > 0
    return keep_mask_256

def apply_keep_mask_nan(arr2d, keep_mask_256, out_size):
    """
    Resize arr2d to out_size, resize keep mask nearest, apply NaN to masked region.
    """
    arr_rs = cv2.resize(arr2d.astype(np.float32), (out_size, out_size), interpolation=cv2.INTER_LINEAR)
    km_rs  = cv2.resize(keep_mask_256.astype(np.uint8), (out_size, out_size),
                        interpolation=cv2.INTER_NEAREST).astype(bool)
    out = arr_rs.copy()
    out[~km_rs] = np.nan
    return out

# -----------------------
# Load models
# -----------------------
models, meta = {}, {}
for name, ckpt_path in CKPTS.items():
    m, info = load_model(ckpt_path)
    models[name] = m
    meta[name] = info

IMG_SIZE = meta["Raw"]["img_size"]  # typically 200

# -----------------------
# Pick patient common to all folders
# -----------------------
common = get_common_filenames(VF_PATHS)
assert len(common) > 0, "No common .npz files found across VF_PATHS!"
assert 0 <= PATIENT_IDX < len(common), f"PATIENT_IDX out of range (0..{len(common)-1})"

fname = common[PATIENT_IDX]
is_left = "_0_" in fname
print(f"Patient: {fname}  ({'Left' if is_left else 'Right'} eye)")

# -----------------------
# Load GT RNFL (from RAW) + flip if left
# -----------------------
raw_data = np.load(RAW_PATH / fname, allow_pickle=True)
rnfl_gt = raw_data["rnflt"].astype(np.float32)
if is_left:
    rnfl_gt = vfskip.flip_rnfl_image(rnfl_gt)

# -----------------------
# Build RAW-derived keep mask (256) and ALSO keep_mask at final size (IMG_SIZE)
# This ensures boundary contour is identical across rows.
# -----------------------
keep_mask_256 = get_keep_mask_from_raw(rnfl_gt)
keep_mask_out = cv2.resize(keep_mask_256.astype(np.uint8), (IMG_SIZE, IMG_SIZE),
                           interpolation=cv2.INTER_NEAREST).astype(bool)
disc_out = (~keep_mask_out).astype(np.uint8)  # 1 inside disc/masked region

print(f"Masked pixels (256): {(~keep_mask_256).sum()} / {keep_mask_256.size} "
      f"({100*(~keep_mask_256).mean():.2f}%)")

# -----------------------
# Load VF vectors
# -----------------------
vf_data = {}
for cond, p in VF_PATHS.items():
    td = np.load(p / fname, allow_pickle=True)["td"].astype(np.float32)
    vf_data[cond] = td.squeeze() if td.ndim == 2 else td

# -----------------------
# Predict RNFL (all conditions)
# -----------------------
pred_um = {name: predict_rnfl_um(models[name], vf_data[name]) for name in CKPTS.keys()}

# Apply RAW-derived mask (NaN -> greyish)
pred_um_m = {k: apply_keep_mask_nan(v, keep_mask_256, out_size=IMG_SIZE) for k, v in pred_um.items()}
rnfl_gt_m = apply_keep_mask_nan(rnfl_gt, keep_mask_256, out_size=IMG_SIZE)

# -----------------------
# Colormaps / norms
# -----------------------
vf_cmap = plt.get_cmap(VF_CMAP_NAME).copy()
vf_cmap.set_bad(VF_NULL_WHITE)
vf_norm = colors.TwoSlopeNorm(vmin=VF_VMIN, vcenter=0.0, vmax=VF_VMAX)

rnfl_cmap = plt.get_cmap(RNFL_CMAP_NAME).copy()
rnfl_cmap.set_bad(RNFL_MASK_GREY)
rnfl_norm = colors.Normalize(vmin=0.0, vmax=RNFL_VMAX)

# -----------------------
# FIGURE
# -----------------------
row_names = list(CKPTS.keys())
n_rows = len(row_names) + 1  # + GT row

fig = plt.figure(figsize=(7.4, 2.85 * n_rows))
gs = gridspec.GridSpec(
    n_rows, 2,
    height_ratios=[1.0]*len(row_names) + [1.05],
    hspace=0.28,
    wspace=0.10,
    top=0.93,
    bottom=0.10,
    left=0.12,
    right=0.98
)

axs = [[None, None] for _ in range(n_rows)]
for r in range(len(row_names)):
    axs[r][0] = fig.add_subplot(gs[r, 0])
    axs[r][1] = fig.add_subplot(gs[r, 1])
axs[-1][0] = fig.add_subplot(gs[-1, :])  # GT spans both cols
axs[-1][1] = None

axs[0][0].set_title("VF Input", fontsize=12, fontweight="bold", pad=6)
axs[0][1].set_title("Predicted RNFL", fontsize=12, fontweight="bold", pad=6)

for r, name in enumerate(row_names):
    axs[r][0].text(
        -0.10, 0.5, name,
        transform=axs[r][0].transAxes,
        fontsize=11.5, fontweight="bold",
        va="center", ha="right"
    )

    # VF
    vf_mat = vf52_to_8x9(vf_data[name])
    axs[r][0].imshow(vf_mat, cmap=vf_cmap, norm=vf_norm,
                     interpolation="nearest", aspect="equal")
    axs[r][0].set_aspect("equal", adjustable="box")
    _no_ticks(axs[r][0])

    # Pred RNFL (masked)
    axs[r][1].imshow(pred_um_m[name], cmap=rnfl_cmap, norm=rnfl_norm,
                     interpolation="nearest", origin="lower", aspect="equal")
    # CONSISTENT boundary from RAW mask at final resolution
    axs[r][1].contour(disc_out, levels=[0.5], colors=[BOUNDARY_COLOR], linewidths=BOUNDARY_LW)
    axs[r][1].set_aspect("equal", adjustable="box")
    _no_ticks(axs[r][1])

# GT full width (masked)
axs[-1][0].set_title("Ground Truth RNFL", fontsize=12, fontweight="bold", pad=6)
axs[-1][0].imshow(rnfl_gt_m, cmap=rnfl_cmap, norm=rnfl_norm,
                  interpolation="nearest", origin="lower", aspect="equal")
axs[-1][0].contour(disc_out, levels=[0.5], colors=[BOUNDARY_COLOR], linewidths=BOUNDARY_LW)
axs[-1][0].set_aspect("equal", adjustable="box")
_no_ticks(axs[-1][0])

# -----------------------
# Colorbars
# -----------------------
cax_vf = fig.add_axes([0.12, 0.05, 0.36, 0.018])
sm_vf = plt.cm.ScalarMappable(cmap=vf_cmap, norm=vf_norm); sm_vf.set_array([])
cb_vf = fig.colorbar(sm_vf, cax=cax_vf, orientation="horizontal",
                     ticks=[VF_VMIN, 0, VF_VMAX])
cb_vf.ax.tick_params(labelsize=9)
cb_vf.set_label("TD (dB)", fontsize=9)

cax_r = fig.add_axes([0.54, 0.05, 0.44, 0.018])
sm_r = plt.cm.ScalarMappable(cmap=rnfl_cmap, norm=rnfl_norm); sm_r.set_array([])
cb_r = fig.colorbar(sm_r, cax=cax_r, orientation="horizontal",
                    ticks=[0, RNFL_VMAX/2, RNFL_VMAX])
cb_r.ax.tick_params(labelsize=9)
cb_r.set_label("RNFL thickness (µm)", fontsize=9)

# Save
pat_id = fname.replace(".npz", "")
outpath = OUT_DIR / f"{pat_id}_vf2rnfl_whiteVF_RAWmask_grey_contour.png"
plt.savefig(outpath, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", outpath.resolve())

In [ ]:
import os
from pathlib import Path
DATA_ROOT = Path(os.environ.get("VF_DATA_ROOT", "/path/to/vf_oct_pairs"))   # see config/env.example.sh


# ==========================================
# Figure: VF Severity × Method Grid
#
# Layout  : 4 rows × 3 columns
#   Rows  : Raw | NAFNet | Noise2Void | CNN-VAE
#   Cols  : Mild | Moderate | Severe
#   Panels: VF heatmap only  (bwr_r, −38…26 dB)
#
# Adjust SEVERITY_PATIENT_INDICES to pick different patients.
# ==========================================

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import colors
from matplotlib.patches import Rectangle
from pathlib import Path

# ---- Paths ----
RAW_PATH = DATA_ROOT / "original_below350"
VF_PATHS = {
    "Raw":        RAW_PATH,
    "NAFNet":     DATA_ROOT / "denoised_NAFNet_models/denoised_NAFNet_Advanced",
    "Noise2Void": DATA_ROOT / "denoised_n2v",
    "CNN-VAE":    DATA_ROOT / "denoised_TwoStage_Final/denoised_cnn_vae",
}
SEVERITY_MAP_PATH = Path("splits/severity_mapping.json")
OUT_DIR = Path("figures_out")
OUT_DIR.mkdir(exist_ok=True)

# ---- Patient selection (0 = first sorted file in that severity group) ----
SEVERITY_PATIENT_INDICES = {
    "Mild":     0,
    "Moderate": 0,
    "Severe":   0,
}

VF_VMIN, VF_VMAX = -38.0, 26.0

# ============================================================
# Helpers
# ============================================================
def vf52_to_8x9(tds_52):
    """Map 52 TD values → 8×9 grid; null cells and blind spots → NaN (white)."""
    mat   = np.full((8, 9), np.nan, dtype=np.float32)
    nulls = {0,1,2,7,8,9,10,17,18,34,43,45,54,55,62,63,64,65,70,71}
    k = 0
    for i in range(8):
        for j in range(9):
            if i*9+j not in nulls:
                mat[i, j] = tds_52[k]
                k += 1
    mat[3, 7] = np.nan   # blind spot (superior)
    mat[4, 7] = np.nan   # blind spot (inferior)
    return mat

def _no_ticks(ax):
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for sp in ax.spines.values():
        sp.set_visible(False)

# ============================================================
# Load severity mapping & build per-severity file lists
# ============================================================
with open(SEVERITY_MAP_PATH) as f:
    sev_map = json.load(f)

# Files common to all VF_PATHS AND present in severity map
common = set(f.name for f in RAW_PATH.glob("*.npz"))
for p in VF_PATHS.values():
    common &= set(f.name for f in p.glob("*.npz"))
common &= set(sev_map.keys())
common_sorted = sorted(common)
print(f"Common files with severity info: {len(common_sorted)}")

by_sev = {"Mild": [], "Moderate": [], "Severe": []}
for fname in common_sorted:
    sev = sev_map[fname].get("severity", "")
    if sev in by_sev:
        by_sev[sev].append(fname)
for sev, lst in by_sev.items():
    print(f"  {sev:<10}: {len(lst)} files")

# ============================================================
# Pick one representative patient per severity
# ============================================================
severities = ["Mild", "Moderate", "Severe"]
chosen = {}   # severity -> fname
for sev in severities:
    files = by_sev[sev]
    assert files, f"No files found for severity: {sev}"
    fname = files[SEVERITY_PATIENT_INDICES[sev]]
    chosen[sev] = fname
    md = sev_map[fname].get("md", "?")
    print(f"  {sev:<10}: {fname}  (MD = {md})")

# ============================================================
# Load VF vectors for each (severity, method) pair
# ============================================================
methods = list(VF_PATHS.keys())    # ["Raw", "NAFNet", "Noise2Void", "CNN-VAE"]

vf_grids = {}   # (severity, method) -> 8×9 np.array
for sev, fname in chosen.items():
    for method, p in VF_PATHS.items():
        td = np.load(p / fname, allow_pickle=True)["td"].astype(np.float32)
        td = td.squeeze() if td.ndim == 2 else td
        vf_grids[(sev, method)] = vf52_to_8x9(td)

# ============================================================
# Build figure  (4 rows × 3 cols)
# ============================================================
n_rows, n_cols = len(methods), len(severities)

vf_cmap = plt.get_cmap("bwr_r").copy()
vf_cmap.set_bad("white")
vf_norm = colors.TwoSlopeNorm(vmin=VF_VMIN, vcenter=0.0, vmax=VF_VMAX)

CELL_W, CELL_H = 2.8, 2.8   # inches per cell
fig = plt.figure(figsize=(CELL_W * n_cols + 1.0,
                           CELL_H * n_rows + 0.8))

gs = gridspec.GridSpec(
    n_rows, n_cols,
    hspace=0.18, wspace=0.08,
    top=0.92, bottom=0.10,
    left=0.14, right=0.97,
)
axs = [[fig.add_subplot(gs[r, c]) for c in range(n_cols)] for r in range(n_rows)]

# ---- Column headers (severity) ----
for c, sev in enumerate(severities):
    md = sev_map[chosen[sev]].get("md", None)
    md_str = f"\n(MD = {md:.1f} dB)" if md is not None else ""
    axs[0][c].set_title(f"{sev}{md_str}", fontsize=12, fontweight="bold", pad=6)

# ---- Row labels (method) ----
for r, method in enumerate(methods):
    axs[r][0].text(
        -0.12, 0.5, method,
        transform=axs[r][0].transAxes,
        fontsize=11, fontweight="bold",
        va="center", ha="right",
    )

# ---- Fill panels ----
for r, method in enumerate(methods):
    for c, sev in enumerate(severities):
        ax  = axs[r][c]
        mat = vf_grids[(sev, method)]
        ax.imshow(mat, cmap=vf_cmap, norm=vf_norm,
                  interpolation="nearest", aspect="equal")
        ax.set_aspect("equal", adjustable="box")
        # Blind-spot outline
        ax.add_patch(Rectangle((6.5, 2.5), 1, 2,
                               fill=False, edgecolor="black", lw=0.7))
        _no_ticks(ax)

# ---- Shared colorbar (bottom, centered) ----
cax = fig.add_axes([0.25, 0.04, 0.50, 0.020])
sm  = plt.cm.ScalarMappable(cmap=vf_cmap, norm=vf_norm)
sm.set_array([])
cb = fig.colorbar(sm, cax=cax, orientation="horizontal",
                  ticks=[VF_VMIN, -20, -10, 0, 10, VF_VMAX])
cb.ax.tick_params(labelsize=9)
cb.set_label("TD (dB)", fontsize=10)

# ---- Save ----
outpath = OUT_DIR / "vf_severity_method_grid.png"
plt.savefig(outpath, dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print(f"\nSaved: {outpath.resolve()}")


In [ ]:
import os
from pathlib import Path
DATA_ROOT = Path(os.environ.get("VF_DATA_ROOT", "/path/to/vf_oct_pairs"))   # see config/env.example.sh

import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import random
from pathlib import Path
from tqdm.auto import tqdm

# ==========================================
# 1. Configuration
# ==========================================
NUM_SAMPLES = 5  # Number of patients to visualize

# --- PATHS ---
RAW_PATH = DATA_ROOT / "original_below350"

PATHS = {
    "Noise2Noise":  DATA_ROOT / "denoised_n2n_FINAL",
    "NAFNet":       DATA_ROOT / "denoised_NAFNet_models/denoised_NAFNet_Advanced",
    "Noise2Void":   DATA_ROOT / "denoised_n2v",
    "CNN-AE":       DATA_ROOT / "denoised_TwoStage_Final/denoised_cnn_ae",
    "CNN-VAE":      DATA_ROOT / "denoised_TwoStage_Final/denoised_cnn_vae",
    "PosEnc":       DATA_ROOT / "denoised_TwoStage_Final/denoised_posenc",
    "MLP-VAE":      DATA_ROOT / "denoised_TwoStage_Final/denoised_mlp_vae",
    "Vanilla-AE":   DATA_ROOT / "denoised_TwoStage_Final/denoised_mlp_ae",
}

OUTPUT_DIR = Path("figures_out")
OUTPUT_DIR.mkdir(exist_ok=True)

# Visualization Limits (dB)
VF_VMIN, VF_VMAX = -38, 26 
DIFF_VMIN, DIFF_VMAX = -10, 10  # Range for Difference Map

# ==========================================
# 2. Helpers
# ==========================================
# Fallback map for 52 points (24-2 Visual Field Pattern)
def convertvf2image(vec):
    img = np.full((12, 12), np.nan)
    if len(vec) == 52:
        # Standard 24-2 mapping (row, col)
        # Insert blind spot
        vec = np.insert(vec, 25, np.nan)  
        vec = np.insert(vec, 34, np.nan)
        mask = [
            (0,3),(0,4),(0,5),(0,6),
            (1,2),(1,3),(1,4),(1,5),(1,6),(1,7),
            (2,1),(2,2),(2,3),(2,4),(2,5),(2,6),(2,7),(2,8),
            (3,0),(3,1),(3,2),(3,3),(3,4),(3,5),(3,6),(3,7),(3,8), 
            (4,0),(4,1),(4,2),(4,3),(4,4),(4,5),(4,6),(4,7),(4,8),
            (5,1),(5,2),(5,3),(5,4),(5,5),(5,6),(5,7),(5,8),
            (6,2),(6,3),(6,4),(6,5),(6,6),(6,7),
            (7,3),(7,4),(7,5),(7,6)
        ]
        for i, (r, c) in enumerate(mask):
            if i < len(vec): img[r, c] = vec[i]
    return img

def robust_plot_vf(ax, data_vec, title, vmin, vmax, cmap, show_values=False):
    """Plots a single VF map on a specific matplotlib axis"""
    if data_vec.ndim == 1:
        img = convertvf2image(data_vec)
    else:
        img = data_vec
        
    current_cmap = plt.cm.get_cmap(cmap).copy()
    current_cmap.set_bad(color='white') 
    
    im = ax.imshow(img, cmap=current_cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.axis('off')
    
    if show_values:
        for i in range(img.shape[0]):
            for j in range(img.shape[1]):
                val = img[i, j]
                if not np.isnan(val):
                    # Smart text color for readability
                    color = 'white' if (abs(val) > 20 and 'RdBu' in cmap) or (val < -10 and cmap=='gray') else 'black'
                    ax.text(j, i, f"{val:.0f}", ha='center', va='center', 
                            color=color, fontsize=6)
    return im

def get_common_files():
    """Finds intersection of files across all folders"""
    sets = [set(f.name for f in RAW_PATH.glob("*.npz"))]
    for name, p in PATHS.items():
        if not p.exists(): print(f"Warning: Path not found {p}")
        sets.append(set(f.name for f in p.glob("*.npz")))
    
    common = list(set.intersection(*sets))
    return common

# ==========================================
# 3. Main Loop
# ==========================================
def main():
    print("Finding common patients...")
    common_files = get_common_files()
    
    if not common_files:
        print("No common files found; check the configured paths.")
        return
    
    print(f"Found {len(common_files)} common files. Generating {NUM_SAMPLES} samples.")
    
    # Pick random samples
    selected_files = random.sample(common_files, min(NUM_SAMPLES, len(common_files)))
    
    for filename in tqdm(selected_files, desc="Plotting"):
        patient_id = filename.replace(".npz", "")
        
        # Load Raw
        raw_data = np.load(RAW_PATH / filename, allow_pickle=True)
        raw_vec = raw_data['td'].astype(np.float32) if 'td' in raw_data else raw_data['arr_0']

        models_to_plot = list(PATHS.keys())
        n_rows = len(models_to_plot)
        
        # Setup Figure: Rows = Models, Cols = Raw | Denoised | Diff
        fig, axes = plt.subplots(n_rows, 3, figsize=(10, 3 * n_rows))
        plt.suptitle(f"Patient: {patient_id}", fontsize=16, y=0.98)

        for i, model_name in enumerate(models_to_plot):
            # Load Denoised
            model_path = PATHS[model_name] / filename
            den_data = np.load(model_path, allow_pickle=True)
            den_vec = den_data['td'].astype(np.float32) if 'td' in den_data else den_data['arr_0']
            
            # Flatten if (1, 52) from batch inference
            if den_vec.ndim == 2 and den_vec.shape[0] == 1:
                den_vec = den_vec.squeeze(0)
            
            # Warn if identical (Model failed to learn)
            if np.allclose(raw_vec, den_vec, atol=1e-5):
                print(f"WARNING: {model_name} is IDENTICAL to Raw for {filename}")

            # Calculate Difference (Denoised - Raw)
            diff_vec = den_vec - raw_vec
            
            # --- Plotting ---
            # 1. Raw Input
            robust_plot_vf(axes[i, 0], raw_vec, "Raw Input", VF_VMIN, VF_VMAX, "gray", show_values=True)
            
            # 2. Denoised Output
            robust_plot_vf(axes[i, 1], den_vec, f"{model_name}", VF_VMIN, VF_VMAX, "gray", show_values=True)
            
            # 3. Difference Map (RdBu: Red=Added, Blue=Removed)
            im = robust_plot_vf(axes[i, 2], diff_vec, f"Diff ({model_name} - Raw)", 
                                DIFF_VMIN, DIFF_VMAX, "RdBu_r", show_values=False)

        # Add shared colorbar for diff maps
        cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7]) 
        fig.colorbar(im, cax=cbar_ax, label="dB Change (Red=Improved, Blue=Worsened)")
        
        plt.tight_layout(rect=[0, 0, 0.9, 0.95])
        
        # Save
        save_path = OUTPUT_DIR / f"{patient_id}_compare.png"
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        plt.close()

    print(f"Comparison figures saved to: {OUTPUT_DIR.resolve()}")

if __name__ == "__main__":
    main()